In [27]:
import os
import torch
import mdtraj as md
import numpy as np
from tqdm import tqdm
from pathlib import Path
from datasets.dataset_utils_empty import Molecules, get_dataset
from models.graph_transformer import GraphTransformer
from evaluate.evaluators import TicEvaluator
from evaluate.msm_utils import discretize_trajectory

device = torch.device(torch.cuda.current_device() if torch.cuda.is_available() else 'cpu')


CLUSTER_ENDPOINTS = {
   'chignolin': [5, 17],
   'trp_cage': [14, 13],
   'bba': [18, 3],
   'villin': [13, 6],
   'protein_g': [17, 2],
}

protein_name = "chignolin"
gen_mode = "langevin" # Mode of generation, "iid" or "langevin"
subsample = None # Give integer if only a random subset of the samples should be analyzed (if None, all samples are used)
append_exp_name = None # Append string to the experiment name"

start = CLUSTER_ENDPOINTS[protein_name][0] - 1
end = CLUSTER_ENDPOINTS[protein_name][1] - 1
append_exp_name_str = '_' + append_exp_name if append_exp_name else ''
eval_folder = f"./saved_models/{protein_name}/main_eval_output_{gen_mode}{append_exp_name_str}"
sample_path = Path(eval_folder, f"sample-{gen_mode}.pt")
pdb_file = f"./datasets/folded_pdbs/{Molecules[protein_name.upper()].value}-0-c-alpha.pdb"

# Load sampled molecules
sampled_mol = torch.load(sample_path)
if subsample is not None:
    sampled_mol = sampled_mol[np.random.permutation(subsample)]
# print(f"Size of samples set (num_samples x num_backbone_atoms x 3): {sampled_mol.shape}")
n_atoms = sampled_mol.shape[1]

# Load topology from pdb file
topology = md.load(pdb_file).topology

# Load cluster centers
cluster_centers_path = Path(
    os.path.join(
        "evaluate",
        "saved_references",
        f"saved_cluster_centers_{protein_name.upper()}.npy",
    )
)
cluster_coords = np.load(cluster_centers_path)

iid_sample_path = Path(
        os.path.join(os.path.dirname(eval_folder), "main_eval_output_iid")
    )
# Get TICA
tic_evaluator = TicEvaluator(
    val_data=None,
    mol_name=protein_name,
    eval_folder=iid_sample_path,
    data_folder="datasets",
    folded_pdb_folder="datasets/folded_pdbs",
    bins=101,
    evalset="testset",
)
# assign cluster centers to the iid samples
cluster_assignments = torch.tensor(discretize_trajectory(
    sampled_mol, tic_evaluator, cluster_coords
))

/home/sanjeevr/om-diffusion/two-for-one-diffusion/evaluate/msm_utils.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(xyz), tic_evaluator.folded


In [34]:
# trainset, _, _ = get_dataset(
#         args.mol,
#         args.mean0,
#         args.data_folder,
#         args.fold,
#         shuffle_before_splitting=args.shuffle_data_before_splitting,
#     )

model = GraphTransformer(num_beads = n_atoms, hidden_nf=64, conservative=True).to(device)

# add sigmoid activation to the output
class CommittorNN(torch.nn.Module):
    def __init__(self, model):
        super(CommittorNN, self).__init__()
        self.model = model
        self.sigmoid = torch.nn.Sigmoid()
    def forward(self, x, h, t):
        committor_prob = self.sigmoid(self.model(x, h, t, return_energy = True))
        return committor_prob


committor_model = CommittorNN(model)

# Train the model on the sampled molecules
committor_model.train()
optimizer = torch.optim.Adam(committor_model.parameters(), lr=3e-4)
batch_size = 256

for i, (x, cluster) in enumerate(zip(sampled_mol.split(batch_size), cluster_assignments.split(batch_size))):
    optimizer.zero_grad()
    x = x.to(device).requires_grad_(True)
    cluster = cluster.to(device)
    start_mask = cluster == start
    end_mask = cluster == end
    transition_mask = torch.logical_and(~start_mask, ~end_mask)
    h = torch.eye(n_atoms).to(device)
    t = torch.zeros((batch_size, )).to(device)
    prob = committor_model(x, h, t)
    grad_prob = torch.autograd.grad(
        outputs=prob,  # [n_graphs, ]
        inputs=x,  # [n_nodes, 3]
        grad_outputs=torch.ones_like(prob),
        retain_graph=True,  # Make sure the graph is not destroyed during training
        create_graph=True,  # Create graph for second derivative
        allow_unused=True,
        )[0]
    grad_loss = 0
    start_boundary_loss = 0
    end_boundary_loss = 0

    if transition_mask.sum() > 0:
        grad_loss = torch.norm(grad_prob, dim=(-2, -1))[transition_mask].mean()
    if start_mask.sum() > 0:
        start_boundary_loss = (prob**2)[start_mask].mean()
    if end_mask.sum() > 0:
        end_boundary_loss = ((1 - prob)**2)[end_mask].mean()

    loss = grad_loss + start_boundary_loss + end_boundary_loss

    loss.backward()
    optimizer.step()
    if i % 100 == 0:
        print(f"step {i}, loss: {loss.item()}")




step 0, loss: 2.5773203372955322
step 100, loss: 0.7166845202445984
step 200, loss: 0.10967987775802612
step 300, loss: 0.05632331967353821
step 400, loss: 0.6070050597190857
step 500, loss: 0.00450570322573185
step 600, loss: 0.020904092118144035
step 700, loss: 0.49244293570518494
step 800, loss: 0.06410755962133408
step 900, loss: 0.02353295311331749
step 1000, loss: 0.043966080993413925
step 1100, loss: 0.5222272872924805
step 1200, loss: 0.046703729778528214
step 1300, loss: 0.125473290681839
step 1400, loss: 0.07236640155315399
step 1500, loss: 0.022131485864520073
step 1600, loss: 0.07848663628101349
step 1700, loss: 0.00040530116530135274
step 1800, loss: 0.000259327789535746
step 1900, loss: 0.09946980327367783
step 2000, loss: 0.05351008474826813
step 2100, loss: 0.4531627297401428
step 2200, loss: 0.5898063778877258
step 2300, loss: 0.041505005210638046
step 2400, loss: 0.10696995258331299
step 2500, loss: 0.49421268701553345
step 2600, loss: 0.08744107931852341
step 2700, l

KeyboardInterrupt: 

In [ ]:
# create a TIC plot and color by predicted committor probability